In [2]:
import sys
import os
import pathlib
import click
import hydra
import torch
import dill
import yaml
import re
import numpy as np
import random
from termcolor import colored
from diffusion_policy.workspace.base_workspace import BaseWorkspace
from omegaconf import OmegaConf
from diffusion_policy.dataset.base_dataset import BaseImageDataset

def find_best_checkpoint(checkpoint_dir):
    ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.ckpt')]
    best_ckpt = None
    best_score = -float('inf')

    pattern = re.compile(r'epoch=\d+-test_mean_score=([\d.]+)\.ckpt')
    for f in ckpt_files:
        match = pattern.match(f)
        if match:
            score = float(match.group(1))
            if score > best_score:
                best_score = score
                best_ckpt = f

    if best_ckpt is None:
        raise FileNotFoundError(
            colored(f"No valid checkpoint found in {checkpoint_dir} matching pattern 'epoch=...-test_mean_score=....ckpt'", "red")
        )
    
    return os.path.join(checkpoint_dir, best_ckpt)


def get_base_log_path(base_dir, policy, nfe):
    """Return the base path without version suffix: eval_log_p1_n2.yaml"""
    base_dir = pathlib.Path(base_dir)
    stem = f"eval_log_p{policy}_n{nfe}"
    return base_dir / (stem + ".yaml")


def to_python_scalar(value):
    if isinstance(value, torch.Tensor):
        if value.numel() == 1:
            # print("This is Tensor")
            return value.item()
        else:
            return None
    if isinstance(value, np.ndarray):
        if value.ndim == 0 or (value.ndim == 1 and value.size == 1):
            # print("This is ndarray")
            return value.item()
        else:
            return None
    if np.isscalar(value):
        try:
            # print("This is isscalar")
            return float(value) if isinstance(value, (np.floating, float)) else int(value)
        except (TypeError, ValueError):
            return None
    if isinstance(value, (int, float, bool)) or value is None:
        print("This is float")
        return value
    return None


checkpoint_dir = "result/square_mh/run_0/checkpoints"
policy = 0
nfe = 0.5
device = 'cuda:1'

checkpoint_dir = pathlib.Path(checkpoint_dir).resolve()
if not checkpoint_dir.is_dir():
    raise NotADirectoryError(colored(f"checkpoint_dir must be a directory: {checkpoint_dir}", "red"))

output_dir = checkpoint_dir.parent / "eval_logs"
output_dir.mkdir(parents=True, exist_ok=True)

# --- Check if evaluation already exists ---
base_log_path = get_base_log_path(output_dir, policy, nfe)
# if base_log_path.exists():
#     print(colored(f"[SKIP] Evaluation already exists at {base_log_path}. Skipping.", "yellow"))
#     return  # Early exit

# --- Proceed with evaluation ---
checkpoint = find_best_checkpoint(str(checkpoint_dir))
print(colored(f"[INFO] Using checkpoint: {checkpoint}", "cyan"))


[INFO] Using checkpoint: /inspire/hdd/project/robot-reasoning/songweixi-240108120099/czt/diffusion_policy/result/square_mh/run_0/checkpoints/epoch=0100-test_mean_score=0.870.ckpt


In [3]:
# print(payload.keys())
# 原始 config

payload = torch.load(open(checkpoint, 'rb'), pickle_module=dill)
cfg = payload['cfg']

OmegaConf.set_readonly(cfg, False)
OmegaConf.set_struct(cfg, False)

print("Using L1 Flow")


cls = hydra.utils.get_class(cfg._target_)
workspace = cls(cfg, output_dir=str(output_dir))
workspace: BaseWorkspace
workspace.load_payload(payload, exclude_keys=None, include_keys=None)

policy_model = workspace.model
if cfg.training.use_ema:
    policy_model = workspace.ema_model


Using L1 Flow


/root/miniforge3/envs/robodiff/lib/python3.9/site-packages/wandb/apis/public.py:2997: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version



============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_gripper_qpos', 'robot0_eef_quat', 'robot0_eef_pos']
using obs modality: rgb with keys: ['robot0_eye_in_hand_image', 'agentview_image']
using obs modality: depth with keys: []
using obs modality: scan with keys: []


/root/miniforge3/envs/robodiff/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and will be removed in 0.15, please use 'weights' instead.
  warnings.warn(
/root/miniforge3/envs/robodiff/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and will be removed in 0.15. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Flow params: 2.556120e+08
Vision params: 2.239418e+07


In [4]:
# 加载 checkpoint → payload2
payload2 = torch.load(open(checkpoint, 'rb'), pickle_module=dill)
cfg2 = payload2['cfg']

# 使用原始 FM 的 inference
cfg2.policy._target_ = "diffusion_policy.policy.flow_test_NFE_origin.FlowUnetL1SampleHybridImagePolicy"
print("Using flow_test_NFE_origin")

# 转成 int（避免 OmegaConf 的 int-like string 问题）
cfg2.policy.num_inference_steps = 2

# 解除 readonly & struct 限制，便于修改
OmegaConf.set_readonly(cfg2, False)
OmegaConf.set_struct(cfg2, False)

# 实例化 workspace2
cls2 = hydra.utils.get_class(cfg2._target_)
workspace2 = cls2(cfg2, output_dir=str(output_dir))
workspace2: BaseWorkspace
workspace2.load_payload(payload2, exclude_keys=None, include_keys=None)

# 获取 policy_model2
policy_model2 = workspace2.model
if cfg2.training.use_ema:
    policy_model2 = workspace2.ema_model

Using flow_test_NFE_origin

============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_gripper_qpos', 'robot0_eef_quat', 'robot0_eef_pos']
using obs modality: rgb with keys: ['robot0_eye_in_hand_image', 'agentview_image']
using obs modality: depth with keys: []
using obs modality: scan with keys: []
Flow params: 2.556120e+08
Vision params: 2.239418e+07


In [5]:
# ====== 🔑 1. 固定随机种子（确保可复现） ======
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ====== 2. 构造共享的最小测试输入（完全一致） ======
# 用 policy_model 的参数定义形状（假设两个模型结构一致，如 horizon/action_dim 相同）
B, T, Da = 1, policy_model.horizon, policy_model.action_dim
device = next(policy_model.parameters()).device  # 或统一用 'cpu' 避免 GPU non-determinism

# 【关键】生成唯一 global_cond，并 clone 到两个模型输入
cond_data = torch.zeros((B, T, Da), device=device)
cond_mask = torch.zeros_like(cond_data, dtype=torch.bool)
global_cond = torch.randn(
    (B, policy_model.n_obs_steps * policy_model.obs_feature_dim),
    device=device
)

# ✅ 显式 clone，避免 inplace 修改影响
global_cond_1 = global_cond.clone()
global_cond_2 = global_cond.clone()

print(f"✅ Input prepared: cond_data={cond_data.shape}, global_cond={global_cond.shape}")

# ====== 3. 确保模型处于 eval 模式（关闭 dropout / BN） ======
policy_model.eval()
policy_model2.eval()

# ====== 4. 分别运行 inference ======
def run_sample(model, cond_data, cond_mask, global_cond, name):
    print(f"\n🚀 Running {name}...")
    try:
        with torch.no_grad():
            sample = model.conditional_sample(
                condition_data=cond_data,
                condition_mask=cond_mask,
                global_cond=global_cond
            )
        print(f"✅ {name} Success! shape: {sample.shape}")
        mean, std = sample.mean().item(), sample.std().item()
        print(f"   mean={mean:.6f}, std={std:.6f}")
        print(f"   first 5 of [0,0]: {sample[0,0,:5].cpu().numpy()}")
        return sample
    except Exception as e:
        print(f"❌ {name} Error:", e)
        raise

sample1 = run_sample(policy_model, cond_data, cond_mask, global_cond_1, "policy_model")
sample2 = run_sample(policy_model2, cond_data, cond_mask, global_cond_2, "policy_model2")

# ====== 5. 比对结果是否一致 ======
print("\n🔍 Comparing outputs...")

# 方法1：逐元素是否全等（要求完全一致，float32 可能因计算路径不同失败）
is_identical = torch.equal(sample1, sample2)
print(f"→ torch.equal (bitwise identical)? {is_identical}")

# 方法2：数值近似相等（推荐）
is_close = torch.allclose(sample1, sample2, atol=1e-6, rtol=1e-5)
print(f"→ torch.allclose (atol=1e-6, rtol=1e-5)? {is_close}")

# 方法3：差异统计
diff = (sample1 - sample2).abs()
print(f"→ Max abs diff: {diff.max().item():.2e}")
print(f"→ Mean abs diff: {diff.mean().item():.2e}")

if not is_close:
    print("⚠️  Warning: Models produce different outputs!")
else:
    print("✅ Models produce numerically identical outputs.")

✅ Input prepared: cond_data=torch.Size([1, 16, 10]), global_cond=torch.Size([1, 274])

🚀 Running policy_model...
✅ policy_model Success! shape: torch.Size([1, 16, 10])
   mean=0.211555, std=0.683887
   first 5 of [0,0]: [ 1.6888629  -0.3696621   0.446692   -0.7505772  -0.55186427]

🚀 Running policy_model2...
✅ policy_model2 Success! shape: torch.Size([1, 16, 10])
   mean=0.209933, std=0.684352
   first 5 of [0,0]: [ 1.6897491  -0.37173197  0.44164187 -0.7512233  -0.5482109 ]

🔍 Comparing outputs...
→ torch.equal (bitwise identical)? False
→ torch.allclose (atol=1e-6, rtol=1e-5)? False
→ Max abs diff: 3.56e-02
→ Mean abs diff: 4.39e-03
⚠️  Warning: Models produce different outputs!


In [ ]:
# configure dataset
dataset: BaseImageDataset
dataset = hydra.utils.instantiate(cfg.task.dataset)
assert isinstance(dataset, BaseImageDataset)
normalizer = dataset.get_normalizer()

# self.model.set_normalizer(normalizer)
# if cfg.training.use_ema:
#     self.ema_model.set_normalizer(normalizer)

# 然后传给 policy model（关键！）
# policy_model.set_normalizer(normalizer)   # ← 必须手动调用！
# seed = cfg.training.seed  # 从checkpoint配置获取
# torch.manual_seed(seed)
# np.random.seed(seed)
# random.seed(seed)


def torch_modules_equal(mod1, mod2):
    sd1 = mod1.state_dict()
    sd2 = mod2.state_dict()
    if sd1.keys() != sd2.keys():
        return False
    return all(torch.allclose(sd1[k], sd2[k]) for k in sd1.keys())

torch_modules_equal(normalizer, policy_model.normalizer)

In [8]:
def conditional_sample(model, 
    condition_data):

    nfe = 2
    # set step values
    x_t = torch.randn(
        size=condition_data.shape, 
        dtype=condition_data.dtype,
        device=condition_data.device)
    # set step values
    dt = 1/nfe
    t = torch.zeros(1, device=condition_data.device)

    for i in range(nfe):
        # 2. predict model output
        a_pred = model(x_t, t)
        v_t = (a_pred - x_t)/(1 - t + 1e-8)

        # 3. compute previous image: x_t -> x_t+dt
        x_t = x_t + dt*v_t
        t = t + dt

    return x_t


conditional_sample(policy_model, cond_data)

NotImplementedError: Module [FlowUnetL1SampleHybridImagePolicy] is missing the required "forward" function